Install Packages

In [1]:
!python -m pip install -qq pandas xarray matplotlib netcdf4 pyproj pyrsig pycno

Import Packages

In [2]:
# Import Libraries
import pyproj
import xarray as xr
import pyrsig
import pandas as pd
import pycno
import getpass
import matplotlib.pyplot as plt

Set location name, spatial extent, and time interval

In [3]:
grid = dict(
    GDNAM='GLOBAL', GDTYP=1, NCOLS=7200, NROWS=3600,
    XORIG=-125, YORIG=20, XCELL=0.01, YCELL=0.01,
    P_ALP=0., P_BET=0., P_GAM=0., XCENT=0., YCENT=0.
)

In [4]:
locname = 'HRBT'
bbox = (-76.93, 36.55, -75.86, 37.37)
bdate = '2024-09-11 10:00:00'
edate = '2024-09-12 19:00:00'

Use Api to get Tempo data

In [5]:
api = pyrsig.RsigApi(bdate=bdate, bbox=bbox, edate=edate, workdir=locname, gridfit=True, grid_kw = grid) #grid_kw = 'global_0pt1') #grid_kw = '1US1')
#api = pyrsig.RsigApi(bdate=bdate, bbox=bbox, workdir=locname, gridfit=True, grid_kw = '1US1')
#api_key = getpass.getpass('Enter TEMPO key (anonymous if unknown):')
api_key = 'anonymous'  # using public, so using anonymous
api.tempo_kw['api_key'] = api_key

In [6]:
print(grid)

{'GDNAM': 'GLOBAL', 'GDTYP': 1, 'NCOLS': 7200, 'NROWS': 3600, 'XORIG': -125, 'YORIG': 20, 'XCELL': 0.01, 'YCELL': 0.01, 'P_ALP': 0.0, 'P_BET': 0.0, 'P_GAM': 0.0, 'XCENT': 0.0, 'YCENT': 0.0}


In [7]:
# after the cell runs, click on the table button.
# Then use filters to find tempo data producs by names that start with tempo
descdf = api.descriptions()
descdf
descdf.query('name.str.contains("tempo.l2.no2.vertical_column_troposphere")')

,name,label,description,bbox_str,beginPosition,timeResolution,endPosition,prefix
5314,tempo.l2.no2.vertical_column_troposphere,l2.no2.vertical_column_troposphere(molecules/cm2),Troposphere nitrogen dioxide vertical column.,-180 -90 180 90,2023-08-02T00:00:00Z,PT1Y,now,tempo
5315,tempo.l2.no2.vertical_column_troposphere_uncer...,l2.no2.vertical_column_troposphere_uncertainty...,Troposphere nitrogen dioxide vertical column u...,-180 -90 180 90,2023-08-02T00:00:00Z,PT1Y,now,tempo


In [8]:
tempokey = 'tempo.l2.no2.vertical_column_troposphere'

In [9]:
tropkey = 'tropomi.offl.no2.nitrogendioxide_tropospheric_column' # This is the tropomi api key

In [10]:
# By default, the pyrsig uses 'ascii' backend, but 'xdr' is faster;
# both look the same in python, but the files are very different.
# I'm using xdr here for speed
df = api.to_dataframe(tempokey, backend='xdr')
#df = api.to_dataframe(tempokey)
df
print(df)

Empty DataFrame
Columns: []
Index: []


In [11]:
# Do it again, but cleanup the keys and add time object
# Notice that the file is reused
df = api.to_dataframe(tempokey, unit_keys=False, parse_dates=True, backend='xdr')
df
print(df)

Using cached: HRBT/tempo.l2.no2.vertical_column_troposphere_2024-09-11T100000Z_2024-09-12T190000Z.xdr.gz
Empty DataFrame
Columns: []
Index: []


In [12]:
# Make an hourly average
hdf = df.groupby(pd.Grouper(key='time', freq='1h')).mean(numeric_only=True)
hdf

KeyError: 'The grouper name time is not found'

In [ ]:
# Plot a data column selected from the names above, Date in title if desired
tempocol = 'no2_vertical_column_troposphere'
tx = hdf[tempocol]
tx.plot(ylabel='NO2 (molec/cm2)', title = 'TEMPO', xlabel='Time (UTC)', color='r',marker='o')

In [ ]:
tx.to_csv('TEMPO'+'_'+bdate[:10]+'_'+edate[:10]+bdate[11:13]+'h'+'_'+edate[11:13]+'h'+'.csv',index=True, mode = 'w')

In [ ]:
tf = api.to_dataframe(tropkey, unit_keys=False, parse_dates=True, backend='xdr')
tf

In [ ]:
tdf = tf.groupby(pd.Grouper(key='time', freq='1h')).mean(numeric_only=True)
tdf

In [ ]:
tropcol = 'nitrogendioxide_tropospheric_column'
trox = tdf[tropcol]
trox.plot(ylabel='NO2 (molec/cm2)', title = 'TROMPOMI', xlabel='Time (UTC)', color='r',marker='o')

In [ ]:
trox.to_csv('TROPOMI'+'_'+bdate[:10]+'_'+edate[:10]+bdate[11:13]+'h'+'_'+edate[11:13]+'h'+'.csv',index=True, mode = 'w')

In [ ]:
nx=df[tempocol]
nx.plot(ylabel='NO2 (molec/cm2)', title = 'TEMPO', xlabel='Time (UTC)', color='y',marker='o')
None
print(nx)

Use Api to get Airnow Data

In [ ]:
airnowkey = 'airnow.no2'
adf = api.to_dataframe(airnowkey, unit_keys=False, parse_dates=True)
adf

In [ ]:
adfh = adf.groupby(pd.Grouper(key='time', freq='1h')).mean(numeric_only=True)

Use Api to get pandora data

In [ ]:
pandorakey = ('pandora.L2_rnvs3p1_8.nitrogen_dioxide_vertical_column_amount')
pdf = api.to_dataframe(pandorakey, unit_keys=False, parse_dates=True)
pdf
type(pdf)

In [ ]:
pdtf = api.to_dataframe(pandorakey, unit_keys=False, parse_dates=True)
pdtf

In [ ]:
#clean data and do hourly average
pdf = pdf.groupby(pd.Grouper(key='time', freq='1h')).mean(numeric_only=True)
pdf

In [ ]:
#plot pandora data
pancol = 'nitrogen_dioxide_vertical_column_amount'
px=pdf[pancol]
px.plot(ylabel='NO2 [molec/cm2]',title='Pandora',xlabel='Time (UTC)',marker='o')
None 
print(px)

In [ ]:
ppx = pdtf[pancol]
ppx.plot(ylabel='NO2 [molec/cm2]',title='Pandora',marker='o', color = 'k')
plt.gca() 
print(ppx)

In [ ]:
ppx = pdtf[pancol]
ppx.plot(ylabel='NO2 [molec/cm2]',title='Pandora',xlabel='Time (UTC)',marker='o', color = 'k')
nx.plot(ylabel='NO2 (molec/cm2)', title = 'TEMPO', xlabel='Time (UTC)', color='y',marker='o')
None 

In [ ]:
dx=pdf[pancol]
dx.plot(ylabel='NO2 [molec/cm2]',title='Pandora',xlabel='Time (UTC)',marker='o')
print(dx)
dx.to_csv('Pandora'+'_'+bdate[:10]+'_'+edate[:10]+bdate[11:13]+'h'+'_'+edate[11:13]+'h'+'.csv',mode='w')
None 

In [ ]:
#plot TEMPO vs Pandora data (same units so dont worry about conversion)
ax.plot(title='TEMPO vs Pandora',ylabel= 'NO2 (molec/cm2)', label='TEMPO', marker='o', linestyle='-',color='r')
px.plot(label='Pandora', marker='o', linestyle='-',xlabel='Time (UTC)',color='b')
#dx.plot(label='Pandora', marker='o', linestyle='-',xlabel='Time (UTC)',color='b')
plt.legend()
None 

In [ ]:
# Plot airnow vs TEMPO, different units, but don't worry, that is why we created the Pandora graph above
airnowno2 = adf['no2'].groupby(adf['time']).median()
ax = hdf[tempocol].plot(ylabel='TEMPO NO2 [molec/cm2]', color='r', marker='+', ylim=(0, 9e15), label = 'TEMPO')
airnowno2.plot(ax=ax.twinx(), color='k', marker='o', ylim=(0, 7), ylabel='AirNow NO2 [ppb]', label = 'AirNow')
plt.legend()
print(airnowno2)
None 

In [ ]:
airnowno2.to_csv('Airnow'+'_'+bdate[:10]+'_'+edate[:10]+bdate[11:13]+'h'+'_'+edate[11:13]+'h'+'.csv',mode='w')

In [ ]:
api.grid_kw
#Set up grid to be mapped, You can see resolution of regrid in GDNAM, in this case it is 1US1, to change, go to grid_kw in first few cells
#change to argument such as 4US1 if desired
#api.grid_kw['XCpELL'].replace('0.01')

In [ ]:
print(api.grid_kw)

In [ ]:
'''
{'GDNAM': 'GLOBAL',
 'GDTYP': 1,
 'NCOLS': 15,
 'NROWS': 13,
 'XORIG': -77.0,
 'YORIG': 36.4,
 'XCELL': 0.1,
 'YCELL': 0.1,
 'P_ALP': 0.0,
 'P_BET': 0.0,
 'P_GAM': 0.0,
 'XCENT': 0.0,
 'YCENT': 0.0,
 'VGTYP': 7,
 'VGTOP': 5000.0,
 'NLAYS': 35,
 'earth_radius': 6370000.0,
 'g': 9.81,
 'R': 287.04,
 'A': 50.0,
 'T0': 290,
 'P0': 100000.0,
 'REGRID_AGGREGATE': 'None'}
 '''

In [ ]:
#build xarray data set for TEMPO NO2
ds = api.to_ioapi(tempokey)
ds

In [ ]:
# approximate coordinate difference 

In [ ]:
# Choose a column from above, notice that names are truncated, so they can be weird
tempoikey = 'NO2_VERTICAL_CO'
long = "LONGITUDE"
lati = "LATITUDE"

In [ ]:
ds[long]

In [ ]:
ds.coords['COL'] = ds.COL.data * ds.XCELL + ds.XORIG
ds.coords['ROW'] = ds.ROW.data * ds.YCELL + ds.YORIG

In [ ]:
# Now plot a map
cno = pycno.cno(ds.crs_proj4)
qm = ds[tempoikey].where(lambda x: x>0).mean(('TSTEP', 'LAY')).plot()
cno.drawstates(resnum=1)

In [ ]:
import xarray as xr
import geopandas as gpd
from shapely.geometry import Point
import rasterio
from rasterio.transform import from_origin

In [ ]:
import rasterio
from rasterio.transform import Affine
import numpy as np

#define georeference points
bottom_left = (-76.930934959, 36.550312014)  # (longitude, latitude)
top_right = (-75.86753207, 37.371761969)  # (longitude, latitude)

#get dimensions
width = ds.sizes['COL']
height = ds.sizes['ROW']

xres = (top_right[0] - bottom_left[0]) / width
yres = (top_right[1] - bottom_left[1]) / height 

#affine transform using bottom left as reference
transform = Affine.translation(bottom_left[0], bottom_left[1]) * Affine.scale(xres, yres)


crs = "EPSG:4326"


data = ds[tempoikey].where(lambda x: x > 0).mean(('TSTEP', 'LAY')).values

#data = np.fliplr(data)

output_path = ('TEMPO_NO2' + bdate[:10] +'_'+bdate[11:13]+'h'+'_'+edate[11:13]+'h'+'.tif')

with rasterio.open(
    output_path,
    'w',
    driver='GTiff',
    height=data.shape[0],
    width=data.shape[1],
    count=1,
    dtype=data.dtype,
    crs=crs,
    transform=transform,
) as dst:
    dst.write(data, 1)

print(f"GeoTIFF saved to {output_path}")